# Simulation Viewer — 2019

Runs the full strategy engine for **25 random trading days in 2019** and shows every detail:
- All 22 strategy signals per stock
- Composite score & ranking
- Quality filter decisions (why a stock passes or fails)
- Position sizing, entry, target, stop loss
- Simulated outcome (WIN / LOSS / TIME_EXIT) and net P&L after costs

**Runtime:** ~10–20 min depending on how many stocks are downloaded for 2019.

Capital: Rs 10,00,000 | Max loss per trade: Rs 20,000 | Max 3 concurrent positions

In [ ]:
import sys, json, random, warnings
from pathlib import Path
from datetime import date, time as dtime

import pandas as pd
import numpy as np
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.2f}'.format)

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

YEAR = 2019
random.seed(42)
print(f'Simulation Viewer — {YEAR}')
print(f'Project root: {project_root}')

In [ ]:
from config.settings import (
    STOCKS_DIR, INDEX_DIR, WEIGHTS_FILE, MAX_RECOMMENDATIONS,
    MAX_LOSS_PER_TRADE, MAX_POSITION_SIZE, DAILY_LOSS_LIMIT,
    CAPITAL,
)
from strategies import ALL_STRATEGIES, STRATEGY_NAMES
from strategies.base import Signal
from backtester.composite_scorer import composite_score, count_agreeing
from backtester.quality_filter import passes_all_filters
from backtester.position_sizer import position_size
from backtester.cost_model import net_pnl
from weights.regime import get_regime_modifiers

# Load weights saved by the engine (from 2016-2018 learning runs)
if WEIGHTS_FILE.exists():
    with open(WEIGHTS_FILE) as f:
        weights = json.load(f)
    print(f'Loaded weights from: {WEIGHTS_FILE}')
else:
    weights = {name: 1.0 for name in STRATEGY_NAMES}
    print('No saved weights — using uniform weights = 1.0')

print(f'\n{"Strategy":<18} {"Weight":>7}  Bar')
print('-' * 50)
for k, v in sorted(weights.items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 8) + '░' * max(0, 24 - int(v * 8))
    print(f'{k:<18} {v:>7.3f}  {bar}')

In [ ]:
# Load 2019 stock data (with 2018 warmup so Jan indicators aren't cold)
# ─────────────────────────────────────────────────────────────────────
try:
    from tqdm.notebook import tqdm as tqdm_nb
except ImportError:
    from tqdm import tqdm as tqdm_nb

all_data = {}
for y in [YEAR - 1, YEAR]:
    yr_dir = STOCKS_DIR / str(y)
    if not yr_dir.exists():
        print(f'  WARNING: {yr_dir} not found — skipping {y}')
        continue
    files = sorted(yr_dir.glob('*.parquet'))
    print(f'Loading {y}: {len(files)} stocks')
    for f in tqdm_nb(files, desc=f'{y}', leave=True):
        try:
            df = pd.read_parquet(f)
            df['datetime'] = pd.to_datetime(df['datetime'])
            stem = f.stem
            if stem in all_data:
                combined = pd.concat([all_data[stem], df])
                combined = (combined.drop_duplicates('datetime')
                                    .sort_values('datetime')
                                    .reset_index(drop=True))
                all_data[stem] = combined
            else:
                all_data[stem] = df
        except Exception:
            pass

# Build trading days list for 2019 only
all_days = set()
for df in all_data.values():
    yr_rows = df[df['datetime'].dt.year == YEAR]
    all_days.update(yr_rows['datetime'].dt.date.unique())
trading_days_2019 = sorted(all_days)

print(f'\nTotal stocks in memory : {len(all_data)}')
print(f'Trading days in {YEAR}   : {len(trading_days_2019)}')
if trading_days_2019:
    print(f'Range                  : {trading_days_2019[0]} → {trading_days_2019[-1]}')

In [ ]:
# Pick 25 random trading days spread across the year
random.seed(42)
n_sample = min(25, len(trading_days_2019))
sample_days = sorted(random.sample(trading_days_2019, n_sample))

print(f'Selected {len(sample_days)} random trading days from {YEAR}:\n')
for i, d in enumerate(sample_days, 1):
    print(f'  {i:2d}.  {d}  ({d.strftime("%A")})')

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def get_today(df, trade_date):
    return df[df['datetime'].dt.date == trade_date].copy().reset_index(drop=True)

def get_prev_day_ohlc(history, trade_date):
    if history.empty:
        return None
    daily = (history.groupby(history['datetime'].dt.date)
             .agg(open=('open','first'), high=('high','max'),
                  low=('low','min'), close=('close','last'),
                  volume=('volume','sum')))
    return daily.iloc[-1] if not daily.empty else None

def estimate_turnover_crore(df, trade_date):
    """20-day avg daily turnover in Crore. Same formula as engine."""
    recent = df[df['datetime'].dt.date < trade_date].tail(20 * 75)
    if recent.empty:
        return 0.0
    daily = (recent.groupby(recent['datetime'].dt.date)
             .apply(lambda x: (x['close'] * x['volume']).sum() / 1e7))
    return float(daily.mean()) if not daily.empty else 0.0

def best_signal(signals, direction=1):
    candidates = [s for s in signals.values() if s.direction == direction and s.is_valid]
    return max(candidates, key=lambda s: s.rr) if candidates else None

def simulate_outcome(signal, today_5min):
    entry, target, stop, dirn = signal.entry, signal.target, signal.stop, signal.direction
    try:
        h, m = map(int, (signal.signal_time or '09:15').split(':'))
        sig_dt = dtime(h, m)
    except Exception:
        sig_dt = dtime(9, 15)
    exit_dt, exit_price, result = dtime(15, 15), entry, 'TIME_EXIT'
    for _, c in today_5min.iterrows():
        t = pd.Timestamp(c['datetime']).time()
        if t <= sig_dt: continue
        if t >= exit_dt:
            exit_price = float(c['open']); result = 'TIME_EXIT'; break
        if dirn == 1:
            if c['high'] >= target:  exit_price = target; result = 'WIN';  break
            if c['low']  <= stop:    exit_price = stop;   result = 'LOSS'; break
            exit_price = float(c['close'])
        elif dirn == -1:
            if c['low']  <= target:  exit_price = target; result = 'WIN';  break
            if c['high'] >= stop:    exit_price = stop;   result = 'LOSS'; break
            exit_price = float(c['close'])
    return exit_price, result

RESULT_LABEL = {'WIN': '✅ WIN', 'LOSS': '❌ LOSS', 'TIME_EXIT': '⏰ TIME_EXIT'}
DIRN_LABEL   = {1: '▲ BUY', -1: '▼ SELL', 0: '─ NEUTRAL'}

print('Helpers loaded.')

In [ ]:
# ── MAIN ANALYSIS LOOP — one rich report per day ──────────────────────────────
# Each day shows 5 sections:
#   1. Top-20 stocks by composite score (overview)
#   2. Signal detail for top-5 candidates (every strategy)
#   3. Quality filter verdict for top-20 candidates
#   4. Final trades selected with position sizing
#   5. Simulated outcome and net P&L

regime_mods  = get_regime_modifiers(weights, vix=15.0)
day_summaries = []

for day_num, trade_date in enumerate(sample_days, 1):

    display(HTML(f'<hr><h2>DAY {day_num}/{len(sample_days)} &nbsp;—&nbsp; {trade_date} &nbsp;({trade_date.strftime("%A")})</h2>'))

    # ── compute signals & scores for every stock ──────────────────────────────
    stock_signals   = {}
    stock_scores    = {}
    stock_turnovers = {}
    stock_today     = {}   # cache today's 5min slice

    for symbol, df in all_data.items():
        today_5min   = get_today(df, trade_date)
        history_5min = df[df['datetime'].dt.date < trade_date]

        if today_5min.empty or len(today_5min) < 5:
            continue

        prev_day = get_prev_day_ohlc(history_5min, trade_date)
        turnover = estimate_turnover_crore(df, trade_date)

        signals = {}
        for strategy in ALL_STRATEGIES:
            try:
                sig = strategy.generate_signal(
                    today_5min=today_5min,
                    history_5min=history_5min,
                    prev_day=prev_day,
                    nifty_today=pd.DataFrame(),
                    trade_date=trade_date,
                )
                signals[strategy.name] = sig
            except Exception:
                signals[strategy.name] = Signal(strategy=strategy.name, direction=0)

        stock_signals[symbol]   = signals
        stock_scores[symbol]    = composite_score(signals, weights, regime_mods)
        stock_turnovers[symbol] = turnover
        stock_today[symbol]     = today_5min

    sorted_stocks = sorted(stock_scores.items(), key=lambda x: x[1], reverse=True)
    display(HTML(f'<b>Active stocks today:</b> {len(sorted_stocks)} &nbsp;|&nbsp; '
                 f'<b>Positive scores:</b> {sum(1 for _, s in sorted_stocks if s > 0)}'))

    # ── SECTION 1: Top-20 overview ────────────────────────────────────────────
    display(HTML('<h3>Section 1 — Top 20 Stocks by Composite Score</h3>'))
    rows = []
    for rank, (sym, score) in enumerate(sorted_stocks[:20], 1):
        sigs = stock_signals[sym]
        buys  = [n for n, s in sigs.items() if s.direction == +1]
        sells = [n for n, s in sigs.items() if s.direction == -1]
        rows.append({
            'Rank':             rank,
            'Stock':            sym,
            'Comp Score':       round(score, 3),
            'BUY  #':           len(buys),
            'SELL #':           len(sells),
            'NEUTRAL #':        len(sigs) - len(buys) - len(sells),
            'Turnover (Cr)':    round(stock_turnovers[sym], 1),
            'Buy strategies':   ', '.join(buys[:6]) + ('…' if len(buys) > 6 else ''),
        })
    display(pd.DataFrame(rows).set_index('Rank'))

    # ── SECTION 2: Per-strategy detail for top 5 ──────────────────────────────
    display(HTML('<h3>Section 2 — Every Strategy Signal for Top 5 Candidates</h3>'))
    for rank, (sym, score) in enumerate(sorted_stocks[:5], 1):
        if score <= 0:
            break
        sigs = stock_signals[sym]
        display(HTML(f'<b>#{rank} {sym}</b> &nbsp; composite score = <b>{score:+.3f}</b> '
                     f'&nbsp; turnover = {stock_turnovers[sym]:.1f} Cr'))
        sig_rows = []
        for strat_name in sorted(sigs.keys()):
            sig = sigs[strat_name]
            if sig.is_valid:
                sig_rows.append({
                    'Strategy':    strat_name,
                    'Signal':      DIRN_LABEL[sig.direction],
                    'Entry ₹':     f'{sig.entry:.2f}',
                    'Target ₹':    f'{sig.target:.2f}',
                    'Stop ₹':      f'{sig.stop:.2f}',
                    'R:R':         f'{sig.rr:.2f}x',
                    'Time':        sig.signal_time or '—',
                    'Reason':      (sig.reason[:60] if sig.reason else '—'),
                })
            else:
                sig_rows.append({
                    'Strategy': strat_name, 'Signal': DIRN_LABEL[sig.direction],
                    'Entry ₹': '—', 'Target ₹': '—', 'Stop ₹': '—',
                    'R:R': '—', 'Time': '—', 'Reason': '—',
                })
        display(pd.DataFrame(sig_rows).set_index('Strategy'))

    # ── SECTION 3: Quality filter verdicts ────────────────────────────────────
    display(HTML('<h3>Section 3 — Quality Filter Decisions (Top 20 Candidates)</h3>'))
    filter_rows = []
    for sym, score in sorted_stocks[:20]:
        sigs     = stock_signals[sym]
        today_5m = stock_today[sym]
        turnover = stock_turnovers[sym]
        b        = best_signal(sigs, direction=+1)

        if b is None:
            filter_rows.append({
                'Stock': sym, 'Score': round(score, 3), 'Best Entry': '—',
                'Liq≥20Cr': '—', 'RR≥1.5': '—', '≥2 Agree': '—',
                'Before2PM': '—', 'VERDICT': '✗ no BUY signal',
            })
            continue

        agreeing     = count_agreeing(sigs, direction=+1)
        passes, why  = passes_all_filters(b, today_5m, turnover, agreeing)

        liq_ok  = '✓' if turnover >= 20.0 else f'✗ {turnover:.1f}Cr'
        rr_ok   = '✓' if b.rr >= 1.5     else f'✗ {b.rr:.2f}'
        agg_ok  = '✓' if agreeing >= 2    else f'✗ {agreeing} strat'
        time_ok = '✓'
        if b.signal_time:
            try:
                h, m = map(int, b.signal_time.split(':'))
                if dtime(h, m) >= dtime(14, 0): time_ok = f'✗ {b.signal_time}'
            except Exception:
                pass

        filter_rows.append({
            'Stock':      sym,
            'Score':      round(score, 3),
            'Best Entry': f'₹{b.entry:.2f}',
            'Liq≥20Cr':   liq_ok,
            'RR≥1.5':     rr_ok,
            '≥2 Agree':   agg_ok,
            'Before2PM':  time_ok,
            'VERDICT':    '✓ SELECTED' if passes else f'✗ {why[:40]}',
        })

    display(pd.DataFrame(filter_rows).set_index('Stock'))

    # ── SECTION 4: Final trades with position sizing ───────────────────────────
    display(HTML('<h3>Section 4 — Final Trades: Position Sizing & Risk</h3>'))

    recommendations = []
    daily_pnl = 0.0

    for sym, score in sorted_stocks[:20]:
        if score <= 0 or len(recommendations) >= MAX_RECOMMENDATIONS:
            break
        sigs     = stock_signals[sym]
        today_5m = stock_today[sym]
        b        = best_signal(sigs, direction=+1)
        if b is None:
            continue
        turnover = stock_turnovers[sym]
        agreeing = count_agreeing(sigs, direction=+1)
        passes, _ = passes_all_filters(b, today_5m, turnover, agreeing)
        if not passes:
            continue

        rs_value, shares = position_size(b.entry, b.stop)
        exit_price, result = simulate_outcome(b, today_5m)
        pnl = net_pnl(b.entry, exit_price, shares)
        daily_pnl += pnl

        stop_loss_rs  = abs(b.entry - b.stop) * shares
        upside_rs     = abs(b.target - b.entry) * shares

        recommendations.append({
            'symbol': sym, 'score': score, 'signal': b,
            'agreeing': agreeing, 'shares': shares, 'investment': rs_value,
            'stop_loss_rs': stop_loss_rs, 'upside_rs': upside_rs,
            'exit_price': exit_price, 'result': result, 'pnl': pnl,
        })

        if daily_pnl < -DAILY_LOSS_LIMIT:
            break

    if recommendations:
        sz_rows = []
        for rec in recommendations:
            b = rec['signal']
            sz_rows.append({
                'Stock':          rec['symbol'],
                'Score':          f"{rec['score']:+.3f}",
                'Agreeing #':     rec['agreeing'],
                'Driver':         b.strategy,
                'Signal @':       b.signal_time or '—',
                'Entry ₹':        f"{b.entry:.2f}",
                'Target ₹':       f"{b.target:.2f}",
                'Stop ₹':         f"{b.stop:.2f}",
                'R:R':            f"{b.rr:.2f}x",
                'Shares':         rec['shares'],
                'Invest ₹':       f"{rec['investment']:,.0f}",
                'Max Loss ₹':     f"{rec['stop_loss_rs']:,.0f}",
                'Max Profit ₹':   f"{rec['upside_rs']:,.0f}",
                'Reason':         (b.reason[:50] if b.reason else '—'),
            })
        display(pd.DataFrame(sz_rows).set_index('Stock'))

        # ── SECTION 5: Simulation outcome ─────────────────────────────────────
        display(HTML('<h3>Section 5 — Simulated Outcome (What Actually Happened)</h3>'))
        out_rows = []
        for rec in recommendations:
            b = rec['signal']
            gross = (rec['exit_price'] - b.entry) * rec['shares'] * (1 if b.direction == 1 else -1)
            out_rows.append({
                'Stock':        rec['symbol'],
                'Entry ₹':      f"{b.entry:.2f}",
                'Exit ₹':       f"{rec['exit_price']:.2f}",
                'Target ₹':     f"{b.target:.2f}",
                'Stop ₹':       f"{b.stop:.2f}",
                'Shares':       rec['shares'],
                'Result':       RESULT_LABEL.get(rec['result'], rec['result']),
                'Gross P&L ₹':  f"{gross:+,.0f}",
                'Net P&L ₹':    f"{rec['pnl']:+,.0f}",
            })
        display(pd.DataFrame(out_rows).set_index('Stock'))
        color = 'green' if daily_pnl >= 0 else 'red'
        display(HTML(f'<b>Day total net P&amp;L: <span style="color:{color}">₹{daily_pnl:+,.0f}</span></b>'))
    else:
        display(HTML('<i>No stocks passed all quality filters today — 0 trades.</i>'))

    day_summaries.append({
        'Date':     str(trade_date),
        'Day':      trade_date.strftime('%a'),
        'Stocks':   len(sorted_stocks),
        'Trades':   len(recommendations),
        'Results':  ' | '.join(r['result'] for r in recommendations) or '—',
        'Net P&L ₹': round(daily_pnl, 2),
        'Cumul ₹':   round(sum(d['Net P&L ₹'] for d in day_summaries) + daily_pnl, 2),
    })

display(HTML('<hr><h2>All 25 Days Complete</h2>'))

In [ ]:
# ── 25-Day Aggregate Summary ──────────────────────────────────────────────────
display(HTML('<h2>25-Day Aggregate Summary — 2019</h2>'))

summary_df = pd.DataFrame(day_summaries).set_index('Date')
display(summary_df)

total_pnl        = sum(d['Net P&L ₹'] for d in day_summaries)
total_trades     = sum(d['Trades']    for d in day_summaries)
days_with_trades = sum(1 for d in day_summaries if d['Trades'] > 0)

all_results = []
for d in day_summaries:
    if d['Results'] != '—':
        all_results.extend(d['Results'].split(' | '))

wins   = all_results.count('WIN')
losses = all_results.count('LOSS')
time_x = all_results.count('TIME_EXIT')

print('\n' + '='*55)
print(f'  Simulation: {YEAR} — 25 random trading days')
print('='*55)
print(f'  Total trades           : {total_trades}')
print(f'  Days with ≥1 trade     : {days_with_trades} / 25')
print(f'  WIN (exact target hit) : {wins}')
print(f'  LOSS (stopped out)     : {losses}')
print(f'  TIME_EXIT              : {time_x}')
if total_trades:
    print(f'  Win rate               : {wins/total_trades*100:.1f}%')
print(f'  Total net P&L          : ₹{total_pnl:+,.0f}')
print(f'  Avg per trading day    : ₹{total_pnl/25:+,.0f}')
print('='*55)
print(f'\n  Capital: ₹{CAPITAL:,.0f}  |  Return on 25 days: {total_pnl/CAPITAL*100:+.2f}%')